PHẦN DÙNG CHUNG (CHỈ CHẠY MỘT LẦN)

In [1]:
# Import thư viện dùng chung
import os
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from src.metrics import load_split_data, evaluate_model, find_best_threshold

os.makedirs("../models", exist_ok=True)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [2]:
# Các hàm dùng chung cho cả hai dataset
def build_model(C, class_weight, penalty):
    """StandardScaler + Logistic Regression. Scaler chỉ fit trên dữ liệu train (tránh leakage)."""
    solver = "lbfgs" if penalty == "l2" else "liblinear"   # l1 cần liblinear
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, penalty=penalty, class_weight=class_weight,
                           solver=solver, max_iter=1000, random_state=42),
    )


def oof_prob(model, X, y):
    """Xác suất lớp 1 dự đoán out-of-fold trên train (mỗi mẫu được dự đoán bởi model chưa thấy nó)."""
    return cross_val_predict(model, X, y, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]


def tune_C(X_train, y_train, c_grid, class_weight, penalty):
    """Thử từng C, mỗi C tìm threshold tối ưu F2 bằng CV trên train. Không dùng test."""
    rows = []
    for C in c_grid:
        p = oof_prob(build_model(C, class_weight, penalty), X_train, y_train)
        thre, f2 = find_best_threshold(y_train, p)
        rows.append({"C": C, "best_threshold": thre, "F2_cv": f2})
    table = pd.DataFrame(rows)
    print(table.to_string(index=False))
    return table


def final_eval(dataset, X_train, X_test, y_train, y_test, C, class_weight, penalty):
    """Chọn threshold từ OOF trên train, huấn luyện lại trên toàn bộ train, đánh giá test đúng một lần."""
    p_oof = oof_prob(build_model(C, class_weight, penalty), X_train, y_train)
    thre, f2_cv = find_best_threshold(y_train, p_oof)
    if thre in (0.05, 0.89):
        print(f"⚠ Threshold={thre} nằm sát biên của find_best_threshold, kết quả có thể chưa tối ưu.")

    model = build_model(C, class_weight, penalty).fit(X_train, y_train)
    prob_test = model.predict_proba(X_test)[:, 1]

    result = evaluate_model(y_test, prob_test, threshold=thre,
                            model_name="LogReg", dataset_name=dataset.capitalize())
    result.update({"C": C, "class_weight": str(class_weight), "penalty": penalty, "F2_cv": f2_cv})

    joblib.dump(model, f"../models/logreg_{dataset}.joblib")
    np.save(f"../models/logreg_{dataset}_test_prob.npy", prob_test)   # cho người 4 làm ensemble

    coef = pd.Series(model[-1].coef_[0], index=X_train.columns)
    coef_table = pd.DataFrame({"coef": coef, "odds_ratio_per_1SD": np.exp(coef)}) \
                   .sort_values("coef", key=abs, ascending=False).round(3)
    return result, coef_table

MODEL CHO FULL DATASET (21 FEATURES)

In [3]:
# Load dataset
X_train_f, X_test_f, y_train_f, y_test_f = load_split_data("full")
print(X_train_f.shape, X_test_f.shape)
print(list(X_train_f.columns))            # phải đủ 21 cột
print("Tỉ lệ lớp 1 (train):", y_train_f.mean().round(4))   # ≈ 0.16

(199188, 21) (49797, 21)
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
Tỉ lệ lớp 1 (train): 0.1602


In [4]:
# >>> ĐIỀU CHỈNH (full): chạy cell này nhiều lần, mỗi lần đổi một tham số rồi so sánh cột F2_cv

# ---------------------------------------------------------------
# C: nghịch đảo độ mạnh của regularization (phạt hệ số lớn để tránh overfit)
#    - C nhỏ  -> phạt mạnh: hệ số bị co nhỏ, mô hình đơn giản hơn (nguy cơ underfit)
#    - C lớn  -> phạt yếu: mô hình bám sát dữ liệu train hơn (nguy cơ overfit)
#    Nên thử: lưới theo thang log, ví dụ [0.001, 0.01, 0.1, 1, 10].
#    Nếu C tốt nhất rơi vào rìa lưới (0.001 hoặc 10) thì mở rộng lưới về phía đó
C_GRID_FULL = [0.0001, 0.001, 0.01, 0.1, 1]

# ---------------------------------------------------------------
# CLASS_WEIGHT: cách xử lý mất cân bằng lớp 
#    - "balanced": tự đặt trọng số lớp tỉ lệ nghịch với tần suất, nên lỗi trên
#                  người bệnh bị phạt nặng hơn. Thường tăng Recall, giảm Precision.
#    - None:       mọi mẫu có trọng số bằng nhau. Xác suất đầu ra "trung thực" hơn,
#                  và việc tìm threshold sẽ tự bù cho mất cân bằng.
#    Nên thử: cả hai giá trị, mỗi lần chạy lại cell này và ghi lại F2_cv tốt nhất.
CLASS_WEIGHT_FULL = "balanced"
# ---------------------------------------------------------------
# PENALTY: loại regularization
#    - "l2" (Ridge): co đều các hệ số, giữ lại tất cả đặc trưng. Mặc định, ổn định.
#    - "l1" (Lasso): có thể ép hệ số của đặc trưng yếu về đúng 0, tức tự loại biến.
#                    Hữu ích để xem biến nào thật sự cần ở bộ full (so với bộ selected).
#                    build_model tự chuyển sang solver "liblinear" khi chọn "l1".
#                    Với "l1", C nhỏ sẽ loại nhiều biến hơn, và chạy chậm hơn "l2".
#    Nên thử: "l2" trước, rồi "l1" để so sánh.
PENALTY_FULL = "l2"

# ---------------------------------------------------------------
tune_full = tune_C(X_train_f, y_train_f, C_GRID_FULL, CLASS_WEIGHT_FULL, PENALTY_FULL)

/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasti

     C  best_threshold  F2_cv
0.0001            0.42 0.6259
0.0010            0.42 0.6264
0.0100            0.42 0.6262
0.1000            0.42 0.6263
1.0000            0.42 0.6263


In [5]:
# >>> ĐIỀU CHỈNH (full): điền C có F2_cv cao nhất ở Cell trên
BEST_C_FULL = 0.001

res_full, coef_full = final_eval("full", X_train_f, X_test_f, y_train_f, y_test_f,
                                 C=BEST_C_FULL,
                                 class_weight=CLASS_WEIGHT_FULL,
                                 penalty=PENALTY_FULL)

# Kết quả
print(res_full)

/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasti

{'Model': 'LogReg', 'Dataset': 'Full', 'Threshold': 0.42, 'PR_AUC': 0.4419, 'ROC_AUC': 0.8184, 'F2_Score': 0.6261, 'Recall': 0.8417, 'Precision': 0.3092, 'F1_Score': 0.4523, 'Specificity (TNR)': 0.6414, 'Accuracy': 0.6735, 'TP': 6714, 'FN': 1263, 'FP': 14998, 'TN': 26822, 'C': 0.001, 'class_weight': 'balanced', 'penalty': 'l2', 'F2_cv': 0.6264}


In [6]:
# Hệ số của mô hình
print(coef_full)

                       coef  odds_ratio_per_1SD
GenHlth               0.556               1.743
BMI                   0.489               1.631
Age                   0.450               1.568
HighBP                0.332               1.394
HighChol              0.290               1.336
CholCheck             0.233               1.262
HvyAlcoholConsump    -0.146               0.864
Sex                   0.127               1.136
Income               -0.119               0.888
HeartDiseaseorAttack  0.066               1.068
PhysHlth             -0.046               0.955
Education            -0.037               0.964
NoDocbcCost           0.033               1.034
Stroke                0.032               1.032
DiffWalk              0.028               1.028
MentHlth             -0.018               0.982
PhysActivity         -0.018               0.982
AnyHealthcare         0.016               1.016
Fruits               -0.014               0.986
Veggies              -0.013             

MODEL CHO SELECTED DATASET (13 FEATURES)

In [7]:
X_train_s, X_test_s, y_train_s, y_test_s = load_split_data("selected")
print(X_train_s.shape, X_test_s.shape)
print(list(X_train_s.columns))            # phải đúng 13 cột: GenHlth, HighBP, BMI, HighChol, Age, DiffWalk,
                                          # PhysHlth, HeartDiseaseorAttack, Income, Education, PhysActivity, Stroke, MentHlth
print("Tỉ lệ lớp 1 (train):", y_train_s.mean().round(4)) # 16%

(199188, 13) (49797, 13)
['GenHlth', 'HighBP', 'BMI', 'HighChol', 'Age', 'DiffWalk', 'PhysHlth', 'HeartDiseaseorAttack', 'Income', 'Education', 'PhysActivity', 'Stroke', 'MentHlth']
Tỉ lệ lớp 1 (train): 0.1602


In [8]:
# >>> ĐIỀU CHỈNH (selected): chạy cell này nhiều lần, mỗi lần đổi một tham số rồi so sánh cột F2_cv
# Chỉnh độc lập với bộ full: số biến khác nhau (13 so với 21) nên C tối ưu có thể khác.

# ---------------------------------------------------------------
# C: nghịch đảo độ mạnh của regularization (phạt hệ số lớn để tránh overfit)
#    - C nhỏ  -> phạt mạnh: hệ số bị co nhỏ, mô hình đơn giản hơn (nguy cơ underfit)
#    - C lớn  -> phạt yếu: mô hình bám sát dữ liệu train hơn (nguy cơ overfit)
#    Nên thử: lưới theo thang log. Ở bộ full, C tốt nhất rơi vào rìa dưới (0.001),
#    nên lưới dưới đây có thêm 0.0001 để kiểm tra xem F2_cv còn thay đổi không.
#    Nếu C tốt nhất vẫn nằm ở rìa lưới thì mở rộng về phía đó
#    (ví dụ [0.00001, 0.0001] hoặc [10, 100, 1000]). Nếu nằm giữa lưới thì
#    thu hẹp quanh giá trị đó, ví dụ [0.05, 0.1, 0.2, 0.5].
#    Với 13 biến và ~200 nghìn mẫu, F2_cv thường gần như phẳng giữa các C.
#    Nếu chênh nhau vài phần nghìn thì coi như tương đương và chọn C nhỏ hơn.
C_GRID_SEL = [0.0001, 0.001, 0.01, 0.1, 1, 10]

# ---------------------------------------------------------------
# CLASS_WEIGHT: cách xử lý mất cân bằng lớp (lớp 1 chỉ chiếm ~16%)
#    - "balanced": tự đặt trọng số lớp tỉ lệ nghịch với tần suất, nên lỗi trên
#                  người bệnh bị phạt nặng hơn. Thường tăng Recall, giảm Precision.
#    - None:       mọi mẫu có trọng số bằng nhau. Xác suất đầu ra "trung thực" hơn,
#                  và việc tìm threshold sẽ tự bù cho mất cân bằng.
#    Nên thử: cả hai giá trị, mỗi lần chạy lại cell này và ghi lại F2_cv tốt nhất.
CLASS_WEIGHT_SEL = "balanced"

# ---------------------------------------------------------------
# PENALTY: loại regularization
#    - "l2" (Ridge): co đều các hệ số, giữ lại tất cả đặc trưng. Mặc định, ổn định.
#    - "l1" (Lasso): có thể ép hệ số của đặc trưng yếu về đúng 0, tức tự loại biến.
#                    build_model tự chuyển sang solver "liblinear" khi chọn "l1".
#                    Với "l1", C nhỏ sẽ loại nhiều biến hơn, và chạy chậm hơn "l2".
#                    Ở bộ selected, 8 biến yếu đã bị loại từ trước, nên nếu l1 vẫn
#                    ép thêm biến nào về 0 thì đó là biến yếu nhất trong 13 biến còn lại.
#    Nên thử: "l2" trước, rồi "l1" để so sánh.
PENALTY_SEL = "l2"

# ---------------------------------------------------------------
tune_sel = tune_C(X_train_s, y_train_s, C_GRID_SEL, CLASS_WEIGHT_SEL, PENALTY_SEL)

/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasti

/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasti

      C  best_threshold  F2_cv
 0.0001            0.42 0.6215
 0.0010            0.41 0.6225
 0.0100            0.40 0.6224
 0.1000            0.40 0.6224
 1.0000            0.40 0.6224
10.0000            0.40 0.6224


In [9]:
# >>> ĐIỀU CHỈNH (selected): điền C có F2_cv cao nhất ở Cell trên
BEST_C_SEL = 0.0010

res_sel, coef_sel = final_eval("selected", X_train_s, X_test_s, y_train_s, y_test_s,
                               C=BEST_C_SEL,
                               class_weight=CLASS_WEIGHT_SEL,
                               penalty=PENALTY_SEL)

# Kết quả
print(res_sel)

/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/run/media/ncn2569/26C2F64BC2F61F2D/261 - HK/261-ML/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasti

{'Model': 'LogReg', 'Dataset': 'Selected', 'Threshold': 0.41, 'PR_AUC': 0.4378, 'ROC_AUC': 0.815, 'F2_Score': 0.6198, 'Recall': 0.8425, 'Precision': 0.3012, 'F1_Score': 0.4438, 'Specificity (TNR)': 0.6272, 'Accuracy': 0.6617, 'TP': 6721, 'FN': 1256, 'FP': 15592, 'TN': 26228, 'C': 0.001, 'class_weight': 'balanced', 'penalty': 'l2', 'F2_cv': 0.6225}


In [10]:
# Hệ số của mô hình
print(coef_sel)

                       coef  odds_ratio_per_1SD
GenHlth               0.564               1.757
BMI                   0.498               1.646
Age                   0.454               1.574
HighBP                0.341               1.407
HighChol              0.295               1.343
Income               -0.102               0.903
HeartDiseaseorAttack  0.081               1.085
PhysHlth             -0.042               0.959
Education            -0.041               0.960
Stroke                0.034               1.035
MentHlth             -0.029               0.971
DiffWalk              0.024               1.025
PhysActivity         -0.013               0.988


BẢNG SO SÁNH GIỮA SAU KHI CHẠY TRÊN HAI DATASET

In [11]:
comparison = pd.DataFrame([res_full, res_sel])
cols = ["Dataset", "C", "class_weight", "penalty", "Threshold", "F2_cv", "F2_Score",
        "Recall", "Precision", "F1_Score", "Specificity (TNR)", "PR_AUC", "ROC_AUC", "TP", "FN", "FP", "TN"]
print(comparison[cols].to_string(index=False))


 Dataset     C class_weight penalty  Threshold  F2_cv  F2_Score  Recall  Precision  F1_Score  Specificity (TNR)  PR_AUC  ROC_AUC   TP   FN    FP    TN
    Full 0.001     balanced      l2       0.42 0.6264    0.6261  0.8417     0.3092    0.4523             0.6414  0.4419   0.8184 6714 1263 14998 26822
Selected 0.001     balanced      l2       0.41 0.6225    0.6198  0.8425     0.3012    0.4438             0.6272  0.4378   0.8150 6721 1256 15592 26228
